# NNUE

## CONFIGURATION

### Imports

In [2]:
import numpy as np
import pandas as pd
import chess
from scipy import sparse

### Constants

In [3]:
INPUT_COUNT = 768
LAYER_1_SIZE = 512
LAYER_2_SIZE = 256
LAYER_3_SIZE = 128
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 0.1  # Reduced from 0.5
MOMENTUM = 0.9

### Key Functions
(Note: Regenerate dataset if labels contain '#' mate scores, as previous logic treated mate as 0.0 eval)

In [4]:
def ReLU(x):
    return np.maximum(0, x)

def Sigmoid(x):
    return 1 / (1 + np.exp(-x))

def ReLU_derivative(x):
    return np.where(x > 0, 1, 0)

def Sigmoid_derivative(x, value = None):
    if (value is not None):
        return value * (1 - value)

    s = Sigmoid(x)
    return s * (1 - s)

# def centipawns_to_win_probability(centipawns : str) -> float:
#     return 1 / (1 + np.exp(-int(centipawns.replace('#', '')) / 400))

def centipawns_to_win_probability(centipawns : str) -> float:
    if '#' in centipawns:
        return 1.0 if '+' in centipawns else 0.0
    return 1 / (1 + np.exp(-int(centipawns) / 400))

def FEN_to_input_vector(fen : str) -> np.ndarray:
    board = chess.Board(fen)
    turn = board.turn 
    
    input_vector = np.zeros(768, dtype=np.float32) # Moins de RAM !
        
    # Optimisation : piece_map() ne boucle que sur les cases occupées (max 32) au lieu de 64
    for square, piece in board.piece_map().items():
        display_square = square if turn == chess.WHITE else chess.square_mirror(square)
        
        piece_type = piece.piece_type - 1
        # Si c'est la pièce du joueur dont c'est le tour, offset = 0, sinon 6
        color_offset = 0 if piece.color == turn else 6               
                    
        input_vector[display_square * 12 + color_offset + piece_type] = 1   
            
    return input_vector

## DATASET 

### Basic Import

In [5]:
labels = np.load('labels.npz')["Y"]
moves = sparse.load_npz('moves.npz')

## Model

### Weights

In [6]:
W1 = np.random.randn(INPUT_COUNT, LAYER_1_SIZE) * np.sqrt(2.0 / INPUT_COUNT)
b1 = np.zeros((1, LAYER_1_SIZE))
W2 = np.random.randn(LAYER_1_SIZE, LAYER_2_SIZE) * np.sqrt(2.0 / LAYER_1_SIZE)
b2 = np.zeros((1, LAYER_2_SIZE))
W3 = np.random.randn(LAYER_2_SIZE, LAYER_3_SIZE) * np.sqrt(2.0 / LAYER_2_SIZE)
b3 = np.zeros((1, LAYER_3_SIZE))
W4 = np.random.randn(LAYER_3_SIZE, 1) * np.sqrt(1.0 / LAYER_3_SIZE)
b4 = np.zeros((1, 1))

vW1, vb1 = np.zeros_like(W1), np.zeros_like(b1)
vW2, vb2 = np.zeros_like(W2), np.zeros_like(b2)
vW3, vb3 = np.zeros_like(W3), np.zeros_like(b3)
vW4, vb4 = np.zeros_like(W4), np.zeros_like(b4)

### Forward Pass

In [7]:
def forward_pass(X, W1, b1, W2, b2, W3, b3, W4, b4):
    
    Z1 = np.dot(X, W1) + b1
    A1 = ReLU(Z1)
    
    Z2 = np.dot(A1, W2) + b2
    A2 = ReLU(Z2)
    
    Z3 = np.dot(A2, W3) + b3
    A3 = ReLU(Z3)
    
    Z4 = np.dot(A3, W4) + b4
    A4 = Sigmoid(Z4)
    
    return A1, Z1, A2, Z2, A3, Z3, A4, Z4

### Backward Pass

In [8]:
def backward_pass(input, W1, b1, A1, Z1, W2, b2, A2, Z2, W3, b3, A3, Z3, W4, b4, A4, Z4, labels):
    m = input.shape[0]

    # Couche 4
    dZ4 = A4 - labels
    dW4 = np.dot(A3.T, dZ4) / m
    db4 = np.sum(dZ4, axis=0, keepdims=True) / m

    # Couche 3
    dA3 = np.dot(dZ4, W4.T)
    dZ3 = dA3 * ReLU_derivative(Z3)
    dW3 = np.dot(A2.T, dZ3) / m  
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m
    
    # Couche 2
    dA2 = np.dot(dZ3, W3.T)
    dZ2 = dA2 * ReLU_derivative(Z2)
    dW2 = np.dot(A1.T, dZ2) / m  
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m
    
    # Couche 1
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * ReLU_derivative(Z1)
    dW1 = np.dot(input.T, dZ1) / m  
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    return dW1, db1, dW2, db2, dW3, db3, dW4, db4

### Training

In [ ]:
for epoch in range(EPOCHS):
    indices = np.arange(moves.shape[0])
    np.random.shuffle(indices)
    total_loss = 0
    
    # Using batches
    for i in range(0, len(indices), BATCH_SIZE):
        batch_indices = indices[i : i + BATCH_SIZE]
        
        # Sparse to dense just for batch
        X_batch = moves[batch_indices].toarray() 
        y_batch = labels[batch_indices].reshape(-1, 1)
        
        # Forward pass (Explicitly passing weights)
        A1, Z1, A2, Z2, A3, Z3, A4, Z4 = forward_pass(X_batch, W1, b1, W2, b2, W3, b3, W4, b4)
        
        # Loss calculation
        epsilon = 1e-15
        loss = -np.mean(y_batch * np.log(A4 + epsilon) + (1 - y_batch) * np.log(1 - A4 + epsilon))
        total_loss += loss
        
        if (i // BATCH_SIZE) % 100 == 0:
            print(f"Epoch {epoch}, Batch {i//BATCH_SIZE}: Loss: {loss:.4f}")
        
        # Backward pass
        dW1, db1, dW2, db2, dW3, db3, dW4, db4 = backward_pass(
            X_batch, W1, b1, A1, Z1, W2, b2, A2, Z2, W3, b3, A3, Z3, W4, b4, A4, Z4, y_batch
        )
        
        vW1 = MOMENTUM * vW1 - LEARNING_RATE * dW1
        vb1 = MOMENTUM * vb1 - LEARNING_RATE * db1
        vW2 = MOMENTUM * vW2 - LEARNING_RATE * dW2
        vb2 = MOMENTUM * vb2 - LEARNING_RATE * db2
        vW3 = MOMENTUM * vW3 - LEARNING_RATE * dW3
        vb3 = MOMENTUM * vb3 - LEARNING_RATE * db3 
        vW4 = MOMENTUM * vW4 - LEARNING_RATE * dW4
        vb4 = MOMENTUM * vb4 - LEARNING_RATE * db4
        
        # Update weights
        W1 += vW1
        b1 += vb1
        W2 += vW2
        b2 += vb2
        W3 += vW3
        b3 += vb3
        W4 += vW4
        b4 += vb4
    
    avg_loss = total_loss / (len(indices) // BATCH_SIZE)
    print(f"End of Epoch {epoch}, Loss: {avg_loss:.4f}")

Epoch 0, Batch 0: Loss: 0.6976
Epoch 0, Batch 100: Loss: 0.6935


KeyboardInterrupt: 

## Full Code

In [ ]:
import numpy as np
import pandas as pd
import chess
from scipy import sparse

INPUT_COUNT = 768
LAYER_1_SIZE = 512
LAYER_2_SIZE = 256
LAYER_3_SIZE = 128
BATCH_SIZE = 1024
EPOCHS = 10
LEARNING_RATE = 0.01 

def ReLU(x):
    return np.maximum(0, x)

def Sigmoid(x):
    return 1 / (1 + np.exp(-x))

def ReLU_derivative(x):
    return np.where(x > 0, 1, 0)

def Sigmoid_derivative(x, value = None):
    if (value is not None):
        return value * (1 - value)

    s = Sigmoid(x)
    return s * (1 - s)

# def centipawns_to_win_probability(centipawns : str) -> float:
#     return 1 / (1 + np.exp(-int(centipawns.replace('#', '')) / 400))

def centipawns_to_win_probability(centipawns : str) -> float:
    if '#' in centipawns:
        return 1.0 if '+' in centipawns else 0.0
    return 1 / (1 + np.exp(-int(centipawns) / 400))

def FEN_to_input_vector(fen : str) -> np.ndarray:
    board = chess.Board(fen)
    turn = board.turn 
    
    input_vector = np.zeros(768, dtype=np.float32) # Moins de RAM !
        
    # Optimisation : piece_map() ne boucle que sur les cases occupées (max 32) au lieu de 64
    for square, piece in board.piece_map().items():
        display_square = square if turn == chess.WHITE else chess.square_mirror(square)
        
        piece_type = piece.piece_type - 1
        # Si c'est la pièce du joueur dont c'est le tour, offset = 0, sinon 6
        color_offset = 0 if piece.color == turn else 6               
                    
        input_vector[display_square * 12 + color_offset + piece_type] = 1   
            
    return input_vector

labels = np.load('labels.npz')["Y"]
moves = sparse.load_npz('moves.npz')

W1 = np.random.randn(INPUT_COUNT, LAYER_1_SIZE) * np.sqrt(2.0 / INPUT_COUNT)
b1 = np.zeros((1, LAYER_1_SIZE))
W2 = np.random.randn(LAYER_1_SIZE, LAYER_2_SIZE) * np.sqrt(2.0 / LAYER_1_SIZE)
b2 = np.zeros((1, LAYER_2_SIZE))
W3 = np.random.randn(LAYER_2_SIZE, LAYER_3_SIZE) * np.sqrt(2.0 / LAYER_2_SIZE)
b3 = np.zeros((1, LAYER_3_SIZE))
W4 = np.random.randn(LAYER_3_SIZE, 1) * np.sqrt(1.0 / LAYER_3_SIZE)
b4 = np.zeros((1, 1))

def forward_pass(X, W1, b1, W2, b2, W3, b3, W4, b4):
    
    Z1 = np.dot(X, W1) + b1
    A1 = ReLU(Z1)
    
    Z2 = np.dot(A1, W2) + b2
    A2 = ReLU(Z2)
    
    Z3 = np.dot(A2, W3) + b3
    A3 = ReLU(Z3)
    
    Z4 = np.dot(A3, W4) + b4
    A4 = Sigmoid(Z4)
    
    return A1, Z1, A2, Z2, A3, Z3, A4, Z4

def backward_pass(input, W1, b1, A1, Z1, W2, b2, A2, Z2, W3, b3, A3, Z3, W4, b4, A4, Z4, labels):
    m = input.shape[0]

    # Couche 4
    dZ4 = A4 - labels
    dW4 = np.dot(A3.T, dZ4) / m
    db4 = np.sum(dZ4, axis=0, keepdims=True) / m

    # Couche 3
    dA3 = np.dot(dZ4, W4.T)
    dZ3 = dA3 * ReLU_derivative(Z3)
    dW3 = np.dot(A2.T, dZ3) / m  
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m
    
    # Couche 2
    dA2 = np.dot(dZ3, W3.T)
    dZ2 = dA2 * ReLU_derivative(Z2)
    dW2 = np.dot(A1.T, dZ2) / m  
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m
    
    # Couche 1
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * ReLU_derivative(Z1)
    dW1 = np.dot(input.T, dZ1) / m  
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    return dW1, db1, dW2, db2, dW3, db3, dW4, db4


for epoch in range(EPOCHS):
    indices = np.arange(moves.shape[0])
    np.random.shuffle(indices)
    total_loss = 0
    
    # Using batches
    for i in range(0, len(indices), BATCH_SIZE):
        batch_indices = indices[i : i + BATCH_SIZE]
        
        # Sparse to dense just for batch
        X_batch = moves[batch_indices].toarray() 
        y_batch = labels[batch_indices].reshape(-1, 1)
        
        # Forward pass (Explicitly passing weights)
        A1, Z1, A2, Z2, A3, Z3, A4, Z4 = forward_pass(X_batch, W1, b1, W2, b2, W3, b3, W4, b4)
        
        # Loss calculation
        epsilon = 1e-15
        loss = -np.mean(y_batch * np.log(A4 + epsilon) + (1 - y_batch) * np.log(1 - A4 + epsilon))
        total_loss += loss
        
        if (i // BATCH_SIZE) % 100 == 0:
            print(f"Epoch {epoch}, Batch {i//BATCH_SIZE}: Loss: {loss:.4f}")
        
        # Backward pass
        dW1, db1, dW2, db2, dW3, db3, dW4, db4 = backward_pass(
            X_batch, W1, b1, A1, Z1, W2, b2, A2, Z2, W3, b3, A3, Z3, W4, b4, A4, Z4, y_batch
        )
        
        # Update weights
        W1 -= LEARNING_RATE * dW1
        b1 -= LEARNING_RATE * db1
        W2 -= LEARNING_RATE * dW2
        b2 -= LEARNING_RATE * db2
        W3 -= LEARNING_RATE * dW3
        b3 -= LEARNING_RATE * db3
        W4 -= LEARNING_RATE * dW4
        b4 -= LEARNING_RATE * db4
    
    avg_loss = total_loss / (len(indices) // BATCH_SIZE)
    print(f"End of Epoch {epoch}, Loss: {avg_loss:.4f}")